# AuksionWatch EDA — 10,000 Lot

10 ta vizualizatsiya + tahlil.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import networkx as nx
import folium
from IPython.display import display

sns.set_theme(style='whitegrid', palette='husl', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

df = pd.read_parquet('../data/lots_features.parquet')
print(f'Yuklandi: {len(df):,} qator, {len(df.columns)} ustun')
df.head(3)

## 1. Viloyat bo'yicha lotlar soni

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
region_counts = df['region_code'].value_counts().sort_values(ascending=True)
region_counts.plot(kind='barh', ax=ax, color=sns.color_palette('Blues_r', len(region_counts)))
ax.set_xlabel('Lotlar soni')
ax.set_title('Viloyat bo\'yicha lotlar soni', fontsize=14, fontweight='bold')
for bar, val in zip(ax.patches, region_counts):
    ax.text(val + 10, bar.get_y() + bar.get_height()/2, f'{val:,}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../data/viz_01_region_counts.png', bbox_inches='tight')
plt.show()

## 2. Ochiq vs Yopiq auksion narx taqsimoti

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

price_data = df[df['sold_price'].notna() & df['auction_type'].notna()].copy()
price_data['price_log'] = np.log1p(price_data['sold_price'])

sns.boxplot(data=price_data, x='auction_type', y='price_log', ax=ax1,
            palette={'open': '#4CAF50', 'closed': '#F44336'})
ax1.set_title('Narx taqsimoti (log shkala)', fontweight='bold')
ax1.set_xlabel('Auksion turi')
ax1.set_ylabel('log(narx)')

type_counts = df['auction_type'].value_counts()
ax2.pie(type_counts, labels=type_counts.index, autopct='%1.1f%%',
        colors=['#4CAF50', '#F44336'], startangle=90)
ax2.set_title('Ochiq vs Yopiq nisbati', fontweight='bold')

plt.tight_layout()
plt.savefig('../data/viz_02_auction_type_price.png', bbox_inches='tight')
plt.show()

## 3. Bidders count gistogrammasi

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
bid_data = df['bidders_count'].dropna()
bid_clipped = bid_data.clip(0, 20)
ax.hist(bid_clipped, bins=21, range=(-0.5, 20.5), color='#2196F3', edgecolor='white', linewidth=0.5)
ax.axvline(1, color='red', linestyle='--', linewidth=1.5, label=f'1 ishtirokchi ({(bid_data==1).mean()*100:.1f}%)')
ax.set_xlabel('Ishtirokchilar soni')
ax.set_ylabel('Lotlar soni')
ax.set_title('Ishtirokchilar soni taqsimoti', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../data/viz_03_bidders_hist.png', bbox_inches='tight')
plt.show()
print(f'1 ishtirokchili lotlar: {(bid_data==1).sum():,} ({(bid_data==1).mean()*100:.1f}%)')

## 4. Discount % gistogrammasi (Red Flag oynasi: >30%)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
disc = df['discount_pct'].dropna()
disc_clipped = disc.clip(-50, 100)

normal = disc_clipped[disc_clipped <= 30]
risky = disc_clipped[disc_clipped > 30]

ax.hist(normal, bins=50, color='#4CAF50', alpha=0.7, label='Normal (≤30%)')
ax.hist(risky, bins=30, color='#F44336', alpha=0.7, label=f'Shubhali (>30%) — {len(risky):,} lot')
ax.axvline(30, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Chegirma %')
ax.set_ylabel('Lotlar soni')
ax.set_title('Chegirma taqsimoti (Red Flag chegarasi: 30%)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../data/viz_04_discount_hist.png', bbox_inches='tight')
plt.show()

## 5. Top 10 sotuvchi (lot soni)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
top_sellers = df.groupby('seller_inn')['lot_id'].count().nlargest(10)
top_sellers_named = df[df['seller_inn'].isin(top_sellers.index)].drop_duplicates('seller_inn').set_index('seller_inn')['seller_name'].fillna('INN: ' + top_sellers.index.astype(str))
labels = [top_sellers_named.get(inn, f'INN:{inn}') for inn in top_sellers.index]

bars = ax.barh(range(len(top_sellers)), top_sellers.values, color=sns.color_palette('Reds_r', 10))
ax.set_yticks(range(len(top_sellers)))
ax.set_yticklabels([str(l)[:30] for l in labels])
ax.set_xlabel('Lotlar soni')
ax.set_title('Top 10 Sotuvchi', fontweight='bold')
for bar, val in zip(bars, top_sellers.values):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2, str(val), va='center')
plt.tight_layout()
plt.savefig('../data/viz_05_top_sellers.png', bbox_inches='tight')
plt.show()

## 6. Top 10 g'olib (yutuq soni)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
winners = df[df['winner_inn'].notna()]
if len(winners) > 0:
    top_winners = winners.groupby('winner_inn')['lot_id'].count().nlargest(10)
    bars = ax.barh(range(len(top_winners)), top_winners.values, color=sns.color_palette('Purples_r', 10))
    ax.set_yticks(range(len(top_winners)))
    ax.set_yticklabels([f'INN: {inn}' for inn in top_winners.index])
    ax.set_xlabel('Yutuq soni')
    ax.set_title('Top 10 G\'olib', fontweight='bold')
    for bar, val in zip(bars, top_winners.values):
        ax.text(val + 0.1, bar.get_y() + bar.get_height()/2, str(val), va='center')
else:
    ax.text(0.5, 0.5, "winner_inn ma'lumoti yo'q", ha='center', va='center', transform=ax.transAxes)
plt.tight_layout()
plt.savefig('../data/viz_06_top_winners.png', bbox_inches='tight')
plt.show()

## 7. Sotuvchi-G'olib Graf (NetworkX, top-50 firma)

In [ ]:
sw = df[df['seller_inn'].notna() & df['winner_inn'].notna()][['seller_inn', 'winner_inn', 'lot_id']]

if len(sw) > 0:
    edges = sw.groupby(['seller_inn', 'winner_inn'])['lot_id'].count().reset_index()
    edges.columns = ['seller', 'winner', 'weight']
    
    # Top 50 firm
    top_nodes = set(
        list(edges.nlargest(50, 'weight')['seller']) +
        list(edges.nlargest(50, 'weight')['winner'])
    )
    edges_top = edges[edges['seller'].isin(top_nodes) & edges['winner'].isin(top_nodes)]

    G = nx.DiGraph()
    for _, row in edges_top.iterrows():
        G.add_edge(row['seller'], row['winner'], weight=row['weight'])

    fig, ax = plt.subplots(figsize=(14, 10))
    pos = nx.spring_layout(G, seed=42, k=2)
    degree = dict(G.degree())
    node_size = [300 + degree.get(n, 1) * 200 for n in G.nodes()]
    edge_width = [G[u][v]['weight'] * 0.3 for u, v in G.edges()]
    
    nx.draw_networkx_nodes(G, pos, node_size=node_size, node_color='#FF7043', alpha=0.8, ax=ax)
    nx.draw_networkx_edges(G, pos, width=edge_width, alpha=0.4, edge_color='gray',
                           arrows=True, arrowsize=10, ax=ax)
    ax.set_title('Sotuvchi → G\'olib Graf (top-50 firma)', fontweight='bold', fontsize=13)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig('../data/viz_07_seller_winner_graph.png', bbox_inches='tight', dpi=100)
    plt.show()
    print(f'Tugunlar: {G.number_of_nodes()}, Qirralar: {G.number_of_edges()}')
else:
    print("Graf uchun yetarli ma'lumot yo'q")

## 8. Geo-xarita (Folium)

In [ ]:
geo_df = df[df['geo_lat'].notna() & df['geo_lon'].notna()].head(2000)

if len(geo_df) > 0:
    m = folium.Map(location=[41.3, 63.9], zoom_start=6, tiles='CartoDB positron')
    
    for _, row in geo_df.iterrows():
        color = 'red' if row.get('is_high_risk', 0) == 1 else 'blue'
        folium.CircleMarker(
            location=[row['geo_lat'], row['geo_lon']],
            radius=4,
            color=color,
            fill=True,
            fill_opacity=0.6,
            popup=folium.Popup(
                f"lot_id={row['lot_id']}<br>type={row.get('lot_type','')}<br>price={row.get('sold_price',''):,}",
                max_width=200
            )
        ).add_to(m)
    
    m.save('../data/viz_08_geo_map.html')
    print(f'{len(geo_df)} lot xaritada ko\'rsatildi → data/viz_08_geo_map.html')
    display(m)
else:
    print("Geo-koordinata ma'lumoti yo'q")

## 9. Sana bo'yicha trend

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
df_time = df[df['start_time'].notna()].copy()

if len(df_time) > 0:
    monthly = df_time.set_index('start_time').resample('ME')['lot_id'].count()
    monthly.plot(ax=ax, color='#1565C0', linewidth=2, marker='o', markersize=4)
    ax.fill_between(monthly.index, monthly.values, alpha=0.2, color='#1565C0')
    ax.set_xlabel('Oy')
    ax.set_ylabel('Lotlar soni')
    ax.set_title('Oylik lot soni trendi', fontweight='bold')
    ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y-%m'))
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('../data/viz_09_time_trend.png', bbox_inches='tight')
    plt.show()
else:
    print("start_time ma'lumoti yo'q")

## 10. Korrelyatsiya matritsasi

In [ ]:
numeric_cols = [
    'start_price', 'sold_price', 'area_m2', 'bidders_count',
    'price_per_m2', 'discount_pct', 'duration_hours',
    'is_closed', 'is_single_bidder', 'red_flag_score',
    'seller_total_lots', 'price_z_score'
]
avail_cols = [c for c in numeric_cols if c in df.columns]
corr = df[avail_cols].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5, ax=ax, fontsize=8)
ax.set_title('Feature Korrelyatsiya Matritsasi', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('../data/viz_10_correlation.png', bbox_inches='tight')
plt.show()

## Xulosa

In [ ]:
print('=' * 50)
print('DATASET STATISTIKASI')
print('=' * 50)
print(f'Jami lotlar:          {len(df):,}')
print(f'Yopiq auksionlar:     {df["auction_type"].eq("closed").sum():,} ({df["auction_type"].eq("closed").mean()*100:.1f}%)')
print(f'1 ishtirokchili:      {df["is_single_bidder"].eq(1).sum():,}')
print(f'Narx anomaliyasi:     {df.get("price_anomaly", pd.Series(0)).eq(1).sum():,}')
print(f'Yuqori risk (>=5):    {df.get("is_high_risk", pd.Series(0)).eq(1).sum():,}')
print(f'Discount >30%:        {(df["discount_pct"] > 30).sum() if "discount_pct" in df.columns else "N/A"}')
print(f'Geo-koordinata bor:   {df["geo_lat"].notna().sum():,}')
print('=' * 50)